## Split data into Training and Test/holdout 

### 1. Import python libraries

In [1]:
import numpy as np
import pandas as pd
import gc
from sklearn.model_selection import train_test_split

### 2. Set options for Jupyter notebook

In [2]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
# Force pandas to display 2 decimal places (no scientific notation)
pd.options.display.float_format = '{:.2f}'.format

### 3. Read the cleaned file

In [3]:
filepath = '../data/interim/neiss_interim_data.parquet'
data_cleaned = pd.read_parquet(filepath)    

In [4]:
data_cleaned.info()

<class 'pandas.DataFrame'>
Index: 7314956 entries, 0 to 7315731
Data columns (total 26 columns):
 #   Column             Dtype         
---  ------             -----         
 0   data_year          int64         
 1   CPSC_Case_Number   int64         
 2   Treatment_Date     datetime64[us]
 3   Age                float64       
 4   Sex                int64         
 5   Race               int64         
 6   Other_Race         str           
 7   Hispanic           float64       
 8   Body_Part          int64         
 9   Diagnosis          int64         
 10  Other_Diagnosis    str           
 11  Body_Part_2        float64       
 12  Diagnosis_2        float64       
 13  Other_Diagnosis_2  str           
 14  Disposition        int64         
 15  Location           int64         
 16  Fire_Involvement   int64         
 17  Product_1          str           
 18  Product_2          int64         
 19  Product_3          int64         
 20  Alcohol            float64       
 21  D

### 4. Perfrom Row Independent Feature Engineering - Derive column based on individual rows. (With out creating Data Leakage) 

#### 4.1 Define the target variable - Hospitalized 

This is needed to do a stratified data split based on treatment year and the disposition code.
Academically, it is said to perfrom feature engineering after the split to avoid data leakage. It is safe to derive new features that are only dependent on individual rows. 

To develop statistically sound predictive models to reliably identify factors associated with hospitalization of product-related injuries; we need to create a target variable for Hospitalized (Yes/No) for each injury reported based on the disposition code.

In [5]:
# B. Define Target Variable (Hospitalization)
# Disposition Codes:
# 1=Released, 2=Transferred, 4=Hospitalized, 5=Held for Obs, 6=Left, 8=Fatality
# We group 2, 4, 5, and 8 as "Severe/Hospitalized"
disposition_codes = [2, 4, 5, 8]
data_cleaned['Hospitalized'] = data_cleaned['Disposition'].apply(lambda x: 1 if x in disposition_codes else 0)
data_cleaned['stratified_split_key'] = data_cleaned['data_year'].astype(str) + '-' + data_cleaned['Hospitalized'].astype(str)

In [6]:
data_cleaned.info()

<class 'pandas.DataFrame'>
Index: 7314956 entries, 0 to 7315731
Data columns (total 28 columns):
 #   Column                Dtype         
---  ------                -----         
 0   data_year             int64         
 1   CPSC_Case_Number      int64         
 2   Treatment_Date        datetime64[us]
 3   Age                   float64       
 4   Sex                   int64         
 5   Race                  int64         
 6   Other_Race            str           
 7   Hispanic              float64       
 8   Body_Part             int64         
 9   Diagnosis             int64         
 10  Other_Diagnosis       str           
 11  Body_Part_2           float64       
 12  Diagnosis_2           float64       
 13  Other_Diagnosis_2     str           
 14  Disposition           int64         
 15  Location              int64         
 16  Fire_Involvement      int64         
 17  Product_1             str           
 18  Product_2             int64         
 19  Product_3       

#### 4.2 Extract Meaningful Information from the narrative flag

In [7]:
narratives = data_cleaned['Narrative_1'].fillna('').str.upper()
regex_patterns = {
        'EMS_Severity': r'\b(AMBULANCE|EMS|PARAMEDIC|UNCONSCIOUS|UNRESPONSIVE|CPR|HELICOPTER|FLIGHT|TRAUMA ALERT)\b',
        'Safety_Gear': r'\b(HELMET|PADS|GOGGLES|GLASSES|SEATBELT|HARNESS|GUARD|PROTECTIVE)\b',
        'Third_Party': r'\b(PUSHED|ASSAULT|FIGHT|BITE|BITTEN|HIT BY|THROWN|ANOTHER PERSON|ALTERCATION)\b',
        'Height_Risk': r'\b(ROOF|LADDER|SCAFFOLD|TREE|STAIRS|BALCONY|WINDOW)\b',
        'Water_Risk': r'\b(POOL|OCEAN|LAKE|BATHTUB|RIVER|HOT TUB|SWIMMING)\b'
    }
data_cleaned['EMS_Severity_Flag'] = np.where(narratives.str.contains(regex_patterns['EMS_Severity'], regex=True), 1, 0)
data_cleaned['Safety_Gear_Flag']  = np.where(narratives.str.contains(regex_patterns['Safety_Gear'], regex=True), 1, 0)
data_cleaned['Third_Party_Flag']  = np.where(narratives.str.contains(regex_patterns['Third_Party'], regex=True), 1, 0)
data_cleaned['Height_Risk_Flag']  = np.where(narratives.str.contains(regex_patterns['Height_Risk'], regex=True), 1, 0)
data_cleaned['Water_Risk_Flag']   = np.where(narratives.str.contains(regex_patterns['Water_Risk'], regex=True), 1, 0)

print("\n--- Feature Extraction Summary (Positive Hits) ---")
print(f"EMS/Severity Involvements: {data_cleaned['EMS_Severity_Flag'].sum():,}")
print(f"Safety Gear Mentions:      {data_cleaned['Safety_Gear_Flag'].sum():,}")
print(f"Third-Party Altercations:  {data_cleaned['Third_Party_Flag'].sum():,}")
print(f"Height-Related Incidents:  {data_cleaned['Height_Risk_Flag'].sum():,}")
print(f"Water-Related Incidents:   {data_cleaned['Water_Risk_Flag'].sum():,}")

/var/folders/ln/zf609vjj3z1_h7nn_glp_bpc0000gn/T/ipykernel_59386/1456062070.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data_cleaned['EMS_Severity_Flag'] = np.where(narratives.str.contains(regex_patterns['EMS_Severity'], regex=True), 1, 0)
/var/folders/ln/zf609vjj3z1_h7nn_glp_bpc0000gn/T/ipykernel_59386/1456062070.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data_cleaned['Safety_Gear_Flag']  = np.where(narratives.str.contains(regex_patterns['Safety_Gear'], regex=True), 1, 0)
/var/folders/ln/zf609vjj3z1_h7nn_glp_bpc0000gn/T/ipykernel_59386/1456062070.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data_cleaned['Third_Party_Flag']  = np.where(narratives.str.contains(regex_patterns['Third_Party'], regex=Tr


--- Feature Extraction Summary (Positive Hits) ---
EMS/Severity Involvements: 73,643
Safety Gear Mentions:      88,713
Third-Party Altercations:  113,125
Height-Related Incidents:  460,381
Water-Related Incidents:   137,852


### 5. Record Selection 

In [15]:
data_cleaned.groupby(['Hospitalized']).size()

Hospitalized
0    6650119
1     664837
dtype: int64

In [8]:
def temporal_stratified_split(df, year_col='stratified_split_key'):
    """
    Splits the dataset into 70% Train, 15% Eval, and 15% Test, 
    ensuring every single year is represented equally in all three datasets.
    """
    # Note: If you only have a full 'Date' column, extract the year first:
    # df['Treatment_Year'] = pd.to_datetime(df['Treatment_Date']).dt.year

    print(f"Total original records: {len(df):,}")

    # Split off the 15% Test Set
    # We stratify based on the Year column
    df_temp, df_test = train_test_split(
        df, 
        test_size=0.15,     
        stratify=df[year_col], 
        random_state=42 # Random state ensures reproducibility
    )

    # Split the remaining 85% into Train (70%) and Eval (15%)
    # Math: 0.15 / 0.85 = 0.17647
    df_train, df_eval = train_test_split(
        df_temp, 
        test_size=(0.15 / 0.85), 
        stratify=df_temp[year_col], 
        random_state=42
    )

    # Verify the distribution
    print("\n--- Split Verification ---")
    print(f"Training Set: {len(df_train):,} rows ({len(df_train)/len(df):.1%})")
    print(f"Eval Set:     {len(df_eval):,} rows ({len(df_eval)/len(df):.1%})")
    print(f"Test Set:     {len(df_test):,} rows ({len(df_test)/len(df):.1%})")
    
    # Prove that the stratification worked (Checking a specific year, e.g., 2020)
    print("\n--- Year Distribution in Train vs Test ---")
    print("Train Years:\n", df_train[year_col].value_counts())
    print("Test Years:\n", df_test[year_col].value_counts())

    return df_train, df_eval, df_test

train_df, eval_df, holdout_df = temporal_stratified_split(data_cleaned, year_col='stratified_split_key')

Total original records: 7,314,956

--- Split Verification ---
Training Set: 5,120,468 rows (70.0%)
Eval Set:     1,097,244 rows (15.0%)
Test Set:     1,097,244 rows (15.0%)

--- Year Distribution in Train vs Test ---
Train Years:
 stratified_split_key
2010-0    263406
2011-0    257017
2012-0    254802
2009-0    254571
2017-0    244791
2008-0    244657
2007-0    242403
2013-0    241922
2006-0    238747
2016-0    238688
2005-0    236967
2014-0    236169
2015-0    229419
2018-0    225755
2019-0    223061
2024-0    221267
2021-0    208390
2023-0    207123
2022-0    198672
2020-0    187255
2024-1     31885
2021-1     29897
2023-1     29635
2020-1     29286
2019-1     28016
2022-1     27652
2018-1     27390
2017-1     26011
2016-1     23927
2015-1     21950
2013-1     21896
2012-1     21240
2014-1     21056
2010-1     20564
2011-1     20500
2009-1     19747
2008-1     17293
2007-1     16439
2006-1     15748
2005-1     15254
Name: count, dtype: int64
Test Years:
 stratified_split_key
2010-0  

In [14]:
holdout_df.groupby(['Hospitalized']).size().reset_index()

,Hospitalized,0
0,0,997518
1,1,99726


In [19]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 5120468 entries, 6181068 to 3926627
Data columns (total 32 columns):
 #   Column             Dtype         
---  ------             -----         
 0   data_year          int64         
 1   CPSC_Case_Number   int64         
 2   Treatment_Date     datetime64[us]
 3   Age                float64       
 4   Sex                int64         
 5   Race               int64         
 6   Other_Race         str           
 7   Hispanic           float64       
 8   Body_Part          int64         
 9   Diagnosis          int64         
 10  Other_Diagnosis    str           
 11  Body_Part_2        float64       
 12  Diagnosis_2        float64       
 13  Other_Diagnosis_2  str           
 14  Disposition        int64         
 15  Location           int64         
 16  Fire_Involvement   int64         
 17  Product_1          str           
 18  Product_2          int64         
 19  Product_3          int64         
 20  Alcohol            float64       


Remove the columns ['stratified_split_key'] column as it is not required for prediction or data analysis.

In [9]:
train_df = train_df.drop(columns=['stratified_split_key'])
eval_df = eval_df.drop(columns=['stratified_split_key'])
holdout_df = holdout_df.drop(columns=['stratified_split_key'])

In [10]:
print(f'The train data has {train_df.shape[0]} rows and {train_df.shape[1]} features')
print(f'The eval data has {eval_df.shape[0]} rows and {eval_df.shape[1]} features')
print(f'The holdout data has {holdout_df.shape[0]} rows and {holdout_df.shape[1]} features')

The train data has 5120468 rows and 32 features
The eval data has 1097244 rows and 32 features
The holdout data has 1097244 rows and 32 features


In [11]:
train_df['Hospitalized'].value_counts()

Hospitalized
0    4655082
1     465386
Name: count, dtype: int64

In [12]:
data_cleaned['Hospitalized'].value_counts()

Hospitalized
0    6650119
1     664837
Name: count, dtype: int64

In [13]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 5120468 entries, 6181068 to 3926627
Data columns (total 32 columns):
 #   Column             Dtype         
---  ------             -----         
 0   data_year          int64         
 1   CPSC_Case_Number   int64         
 2   Treatment_Date     datetime64[us]
 3   Age                float64       
 4   Sex                int64         
 5   Race               int64         
 6   Other_Race         str           
 7   Hispanic           float64       
 8   Body_Part          int64         
 9   Diagnosis          int64         
 10  Other_Diagnosis    str           
 11  Body_Part_2        float64       
 12  Diagnosis_2        float64       
 13  Other_Diagnosis_2  str           
 14  Disposition        int64         
 15  Location           int64         
 16  Fire_Involvement   int64         
 17  Product_1          str           
 18  Product_2          int64         
 19  Product_3          int64         
 20  Alcohol            float64       


In [14]:
eval_df.info()

<class 'pandas.DataFrame'>
Index: 1097244 entries, 1524627 to 3993119
Data columns (total 32 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   data_year          1097244 non-null  int64         
 1   CPSC_Case_Number   1097244 non-null  int64         
 2   Treatment_Date     1097244 non-null  datetime64[us]
 3   Age                1097244 non-null  float64       
 4   Sex                1097244 non-null  int64         
 5   Race               1097244 non-null  int64         
 6   Other_Race         1097244 non-null  str           
 7   Hispanic           1097244 non-null  float64       
 8   Body_Part          1097244 non-null  int64         
 9   Diagnosis          1097244 non-null  int64         
 10  Other_Diagnosis    1097244 non-null  str           
 11  Body_Part_2        1097244 non-null  float64       
 12  Diagnosis_2        1097244 non-null  float64       
 13  Other_Diagnosis_2  1097244 non-null  

In [15]:
holdout_df.info()

<class 'pandas.DataFrame'>
Index: 1097244 entries, 3274389 to 5821132
Data columns (total 32 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   data_year          1097244 non-null  int64         
 1   CPSC_Case_Number   1097244 non-null  int64         
 2   Treatment_Date     1097244 non-null  datetime64[us]
 3   Age                1097244 non-null  float64       
 4   Sex                1097244 non-null  int64         
 5   Race               1097244 non-null  int64         
 6   Other_Race         1097244 non-null  str           
 7   Hispanic           1097244 non-null  float64       
 8   Body_Part          1097244 non-null  int64         
 9   Diagnosis          1097244 non-null  int64         
 10  Other_Diagnosis    1097244 non-null  str           
 11  Body_Part_2        1097244 non-null  float64       
 12  Diagnosis_2        1097244 non-null  float64       
 13  Other_Diagnosis_2  1097244 non-null  

### 4. Copy the data into the Processed Folder as a parquet file.

In [16]:
# Copy the training data into the processed folder
train_df.to_parquet('../data/processed/neiss_clean_train.parquet')
# Copy the evaluation data into the processed folder
eval_df.to_parquet('../data/processed/neiss_clean_eval.parquet')
# Copy the holdout dara into the processed folder
holdout_df.to_parquet('../data/processed/neiss_clean_holdout.parquet')

Data saved in parquet file to reduce the size.

Force run the python garbage collector

In [17]:
n = gc.collect()
print(f"Number of unreachable objects collected: {n}")

Number of unreachable objects collected: 0
